# Study 911 — REIT Quality Screen — the teardown

The excess-Sharpe race, the durable-income tilt (HAC spread *t* + bootstrap Sharpe-advantage CI + era cut), the levered-carry trap, the daily drawdowns, the costed quality book, and the 20-seed synthetic control.

In [1]:
R = {'start': '2007-06', 'end': '2026-06', 'n_months': 229, 'fingerprint': 'b99a4946b405', 'sh_rez': 0.36, 'sh_vnq': 0.29, 'sh_rwr': 0.27, 'sh_rem': 0.039, 'sh_spy': 0.644, 'ann_rez': 6.83, 'ann_vnq': 5.4, 'ann_rwr': 4.95, 'ann_rem': -1.13, 'ann_spy': 10.68, 'vol_rez': 20.9, 'vol_vnq': 22.3, 'vol_rem': 24.2, 'rezvnq_bps': 8.7, 'rezvnq_t': 0.75, 'book_bps': 2.0, 'book_t': 0.41, 'adv': 0.07, 'adv_lo': -0.07, 'adv_hi': 0.202, 'adv_fracneg': 0.17, 'era1_rezvnq_bps': 1.4, 'era1_rezvnq_t': 0.08, 'era2_rezvnq_bps': 16.2, 'era2_rezvnq_t': 1.09, 'rezrem_bps': 55.0, 'rezrem_t': 1.72, 'era1_rezrem_bps': 92.7, 'era1_rezrem_t': 1.83, 'era2_rezrem_bps': 17.1, 'era2_rezrem_t': 0.47, 'dd_rez': -66.9, 'dd_vnq': -73.1, 'dd_rem': -74.7, 'dd_spy': -55.2, 'cost2_net': 1.8, 'cost2_t': 0.37, 'cost2_ann': 0.22, 'cost5_net': 1.5, 'cost5_t': 0.31, 'cost5_ann': 0.18, 'cost10_net': 1.0, 'cost10_t': 0.21, 'cost10_ann': 0.12, 'null_adv_mean': -0.019, 'null_straddle': 17, 'null_trapflag': 20, 'planted_t': 3.17, 'planted_adv': 0.109, 'planted_adv_lo': 0.04}

## The race — excess-vs-excess Sharpe (minus BIL)

In [2]:
for s,(sh,a,v) in {'REZ':(R['sh_rez'],R['ann_rez'],R['vol_rez']),
                   'VNQ':(R['sh_vnq'],R['ann_vnq'],R['vol_vnq']),
                   'REM':(R['sh_rem'],R['ann_rem'],R['vol_rem'])}.items():
    print(f'{s}: excessSharpe {sh:+.3f}  ann {a:+.2f}%  vol {v:.1f}%')

REZ: excessSharpe +0.360  ann +6.83%  vol 20.9%
VNQ: excessSharpe +0.290  ann +5.40%  vol 22.3%
REM: excessSharpe +0.039  ann -1.13%  vol 24.2%


## Leg 1 — the durable-income tilt (REZ vs VNQ): thin, not certified

In [3]:
print(f"REZ-VNQ spread : {R['rezvnq_bps']:+.1f} bps/mo  HAC t = {R['rezvnq_t']:+.2f}")
print(f"quality book   : {R['book_bps']:+.1f} bps/mo  HAC t = {R['book_t']:+.2f}")
print(f"Sharpe adv     : {R['adv']:+.3f}  95% CI [{R['adv_lo']:+.3f}, {R['adv_hi']:+.3f}]  frac<0 = {R['adv_fracneg']:.2f}")
print(f"  era 2007-2016: {R['era1_rezvnq_bps']:+.1f} bps  t = {R['era1_rezvnq_t']:+.2f}")
print(f"  era 2017-2026: {R['era2_rezvnq_bps']:+.1f} bps  t = {R['era2_rezvnq_t']:+.2f}")

REZ-VNQ spread : +8.7 bps/mo  HAC t = +0.75
quality book   : +2.0 bps/mo  HAC t = +0.41
Sharpe adv     : +0.070  95% CI [-0.070, +0.202]  frac<0 = 0.17
  era 2007-2016: +1.4 bps  t = +0.08
  era 2017-2026: +16.2 bps  t = +1.09


## Leg 2 — the leveraged-carry trap (mortgage REITs)

In [4]:
print(f"REZ-REM spread : {R['rezrem_bps']:+.1f} bps/mo  HAC t = {R['rezrem_t']:+.2f}  (GFC-concentrated)")
print(f"  era 2007-2016: {R['era1_rezrem_bps']:+.1f} bps  t = {R['era1_rezrem_t']:+.2f}")
print(f"  era 2017-2026: {R['era2_rezrem_bps']:+.1f} bps  t = {R['era2_rezrem_t']:+.2f}")
print(f"REM total return {R['ann_rem']:+.2f}%/yr vs VNQ {R['ann_vnq']:+.2f}%/yr -- a yield trap on total-return basis")

REZ-REM spread : +55.0 bps/mo  HAC t = +1.72  (GFC-concentrated)
  era 2007-2016: +92.7 bps  t = +1.83
  era 2017-2026: +17.1 bps  t = +0.47
REM total return -1.13%/yr vs VNQ +5.40%/yr -- a yield trap on total-return basis


## Risk — daily total-return max drawdowns

In [5]:
for s,d in [('REZ',R['dd_rez']),('VNQ',R['dd_vnq']),('REM',R['dd_rem']),('SPY',R['dd_spy'])]:
    print(f'{s}: {d:+.1f}%')

REZ: -66.9%
VNQ: -73.1%
REM: -74.7%
SPY: -55.2%


## Tradability — the quality book, costed vs buy-and-hold VNQ

In [6]:
for tag,n,t,a in [('2 bps',R['cost2_net'],R['cost2_t'],R['cost2_ann']),
                  ('5 bps',R['cost5_net'],R['cost5_t'],R['cost5_ann']),
                  ('10 bps',R['cost10_net'],R['cost10_t'],R['cost10_ann'])]:
    print(f'{tag:>6} one-way: net {n:+.1f} bps/mo (t={t:+.2f}, ~{a:+.2f}%/yr)')

 2 bps one-way: net +1.8 bps/mo (t=+0.37, ~+0.22%/yr)
 5 bps one-way: net +1.5 bps/mo (t=+0.31, ~+0.18%/yr)
10 bps one-way: net +1.0 bps/mo (t=+0.21, ~+0.12%/yr)


## Synthetic positive control — the machinery is unbiased

Live: on the null the Sharpe-advantage CI must straddle zero; a planted edge must light up; the trap detector must always flag the inferior-Sharpe leg.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from reit_quality import data, strategy as st
straddle = trap = 0
for s in range(8):
    w = data.synthetic_world(edge_ann=0.0, seed=911+s)
    a = st.sharpe_advantage(w, 'QUAL','BROAD', rf='CASH', n_boot=500)
    straddle += int(a['ci_low'] < 0 < a['ci_high'])
    trap += int(st.excess_sharpe(w,'TRAP','CASH') < st.excess_sharpe(w,'BROAD','CASH'))
print(f'null (edge=0), 8 seeds: CI straddles zero in {straddle}/8; trap flagged {trap}/8')
d = st.synth_detect(data.synthetic_world(edge_ann=0.03, seed=911))
print(f"planted (+3%/yr): QUAL-BROAD HAC t = {d['spread_t']:+.2f}, Sharpe adv = {d['adv']:+.3f} (CI low {d['adv_ci_low']:+.3f})")

null (edge=0), 8 seeds: CI straddles zero in 6/8; trap flagged 8/8
planted (+3%/yr): QUAL-BROAD HAC t = +3.17, Sharpe adv = +0.109 (CI low +0.040)


## Verdict

- **Signal — Mixed.** The durable-income tilt is **not certified**: REZ − VNQ is **+8.7 bps/mo at HAC *t* = 0.75**, Sharpe-advantage CI **[-0.070, +0.202]** straddles zero, not era-robust (t = 0.08 → 1.09). The levered-carry **trap** *is* real: mortgage REITs earned **-1.13%/yr** at Sharpe 0.039 — but the broad index already excludes it.
- **Tradability — Fragile.** The costed quality book nets **~+0.18%/yr at *t* = 0.31** over VNQ (costs barely matter — the gross edge is only ~2 bps/mo); the one robust action (avoid mortgage REITs) is already free inside the broad ETF you'd hold anyway.